In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
# Import tsim from the Bloqade ecosystem as specified in the challenge
import tsim 

def physical_angle(logical_angle_in_pi: float, num_physical_rotations: int) -> float:
    """
    Compute the physical rotation angle needed to achieve a logical rotation of
    angle `logical_angle_in_pi` on `num_physical_rotations` physical rotations.
    """
    assert (
        num_physical_rotations % 2 == 1 and num_physical_rotations > 0
    ), "k must be a positive odd integer"
    
    sign = -1 if (num_physical_rotations + 1) % 4 == 0 else 1
    logical_angle_in_rad = logical_angle_in_pi * np.pi
    x = np.tan(logical_angle_in_rad / 2) ** (1 / num_physical_rotations)
    theta_phys = 2 * np.arctan(x)
    return float(sign * theta_phys / np.pi)

def create_modified_circuit(base_circuit_str: str, logical_angle_pi: float, d: int = 3) -> str:
    """
    Modifies the base STAR circuit string with the new physical and logical angles.
    """
    num_physical_rotations = d
    phys_angle_pi = physical_angle(logical_angle_pi, num_physical_rotations)
    
    # 1. Replace the physical rotation angle (applied to qubits 0, 3, 6)
    # The regex looks for the specific R_Z gate followed by "0 3 6"
    phys_pattern = r"R_Z\([^)]+\)\s+0\s+3\s+6"
    new_phys_gate = f"R_Z({phys_angle_pi}) 0 3 6"
    modified_str = re.sub(phys_pattern, new_phys_gate, base_circuit_str)
    
    # 2. Replace the perfect logical unrotation angle (applied to qubit 6)
    # The regex looks for the R_Z gate applied right before the final CX gates
    log_pattern = r"R_Z\([^)]+\)\s+6"
    # The unrotation is in the opposite direction
    new_log_gate = f"R_Z({-logical_angle_pi}) 6" 
    modified_str = re.sub(log_pattern, new_log_gate, modified_str)
    
    return modified_str

def run_star_simulations(circuit_filepath: str, shots: int = 10000):
    # Load the provided star_d=3.stim file
    with open(circuit_filepath, 'r') as f:
        base_circuit = f.read()
        
    logical_angles_pi = np.linspace(0.01, 0.5, 20)
    logical_fidelities = []
    
    for log_ang in logical_angles_pi:
        # Generate the new circuit string for this angle
        mod_circuit_str = create_modified_circuit(base_circuit, log_ang, d=3)
        
        # Initialize the tsim circuit from the STIM-formatted string
        # (Adjust the specific tsim initialization method to match your Bloqade version)
        circuit = tsim.Circuit.from_string(mod_circuit_str)
        
        # Run the simulation
        # Tsim returns samples for the detectors and the final observable
        results = tsim.sample(circuit, shots=shots)
        
        # --- Post-Selection Logic ---
        # We must post-select on all 24 previous detectors being 0 to guarantee a successful rotation
        # In tsim/stim output arrays, detectors usually precede the observable.
        # Assuming results.detectors has shape (shots, num_detectors) and results.observables has shape (shots, 1)
        
        detectors = results.detectors 
        observables = results.observables[:, 0]
        
        # Find runs where ALL 24 detectors triggered 0 (no errors detected in those rounds)
        # Note: 3 rounds * (d**2 - 1) = 24 detectors
        post_selected_indices = np.where(np.sum(detectors[:, :24], axis=1) == 0)[0]
        
        if len(post_selected_indices) == 0:
            logical_fidelities.append(0.0) # Avoid division by zero if all shots fail
            continue
            
        valid_observables = observables[post_selected_indices]
        
        # If we measure the logical |+>, we should obtain the result 0
        # A result of 1 means a logical error occurred
        success_count = np.sum(valid_observables == 0)
        fidelity = success_count / len(valid_observables)
        
        logical_fidelities.append(fidelity)
        
    return logical_angles_pi, logical_fidelities

# --- Execution and Plotting ---
# Make sure the path points to where you saved the provided star_d=3.stim file
angles, fidelities = run_star_simulations('star_d=3.stim', shots=10000)

plt.figure(figsize=(8, 5))
plt.plot(angles, fidelities, marker='o', linestyle='-', color='b', label='d=3 STAR Fidelity')
plt.title('Logical Fidelity vs. Logical Rotation Angle')
plt.xlabel('Logical Angle (in units of pi)')
plt.ylabel('Logical Fidelity')
plt.grid(True)
plt.legend()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'circuits/star_d=3.stim'